# TensorFlow Basics Tutorial
## Preparation for Week 7: Neural Networks and Vectorization

This is an **executable tutorial**. Run each code cell in order and inspect the output. There are no TODO exercises; the goal is to learn TensorFlow syntax and connect it to the NumPy mathematics used in the lecture.

### Learning goals
By the end, you should understand tensors, shapes and dtypes, NumPy/TensorFlow conversion, matrix multiplication, activation functions, `Dense` layers, `Sequential` models, forward propagation, `compile()`, `fit()`, `predict()`, and vectorization.

## 1. Import TensorFlow and NumPy

TensorFlow is conventionally imported as `tf`. We also use NumPy because the Week 7 practical moves between NumPy and TensorFlow.

In [ ]:
import numpy as np
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

## 2. Creating Tensors

A tensor is a multidimensional array. Its **rank** is the number of dimensions; its **shape** gives the size of each dimension.

In [ ]:
scalar = tf.constant(7.0)
vector = tf.constant([1.0, 2.0, 3.0])
matrix = tf.constant([[1.0, 2.0, 3.0],
                      [4.0, 5.0, 6.0]])
tensor3d = tf.constant([[[1.0, 2.0], [3.0, 4.0]],
                        [[5.0, 6.0], [7.0, 8.0]]])

for name, t in [("scalar", scalar), ("vector", vector),
                ("matrix", matrix), ("3D tensor", tensor3d)]:
    print(f"{name}:")
    print(t)
    print("shape =", t.shape, "| rank =", tf.rank(t).numpy())
    print()

## 3. Data Types

Neural-network calculations commonly use `float32`. TensorFlow does not always automatically mix incompatible data types, so checking `dtype` is useful.

In [ ]:
x_int = tf.constant([1, 2, 3])
x_float = tf.constant([1.0, 2.0, 3.0], dtype=tf.float32)
x_cast = tf.cast(x_int, tf.float32)

print("Integer tensor dtype:", x_int.dtype)
print("Float tensor dtype:", x_float.dtype)
print("After casting:", x_cast, x_cast.dtype)

## 4. NumPy ↔ TensorFlow

TensorFlow accepts NumPy arrays easily, and tensors can be converted back to NumPy with `.numpy()` during eager execution.

In [ ]:
x_np = np.array([[200.0, 17.0],
                 [120.0, 5.0]], dtype=np.float32)

x_tf = tf.convert_to_tensor(x_np)
x_back = x_tf.numpy()

print("NumPy:", type(x_np), x_np.shape)
print(x_np)
print("\nTensorFlow:", type(x_tf), x_tf.shape)
print(x_tf)
print("\nBack to NumPy:", type(x_back), x_back.shape)

## 5. Why We Use 2D Inputs

For tabular neural-network data, we use

$
X:\;(\text{number of samples},\text{number of features}).
$

Thus `[[200.0, 17.0]]` has shape `(1,2)`: **one sample with two features**.

In [ ]:
x_1d = tf.constant([200.0, 17.0])
x_2d = tf.constant([[200.0, 17.0]])
x_reshaped = tf.reshape(x_1d, (1, 2))

X_batch = tf.constant([
    [200.0, 17.0],
    [120.0,  5.0],
    [425.0, 20.0],
    [212.0, 18.0]
])

print("1D:", x_1d.shape)
print("2D:", x_2d.shape)
print("Reshaped:", x_reshaped.shape)
print("Four samples:", X_batch.shape)

## 6. Element-wise Operations vs. Matrix Multiplication

`A * B` is element-wise multiplication. `tf.matmul(A, B)` or `A @ B` is matrix multiplication.

In [ ]:
A = tf.constant([[1., 2.],
                 [3., 4.]])
B = tf.constant([[5., 6.],
                 [7., 8.]])

print("A + B:")
print(A + B)

print("\nA * B (element-wise):")
print(A * B)

print("\nA @ B (matrix multiplication):")
print(A @ B)

## 7. Matrix Multiplication and Neural-Network Shapes

If $X:(m,n)$ and $W:(n,u)$, then $XW:(m,u)$. The inner dimensions must agree.

In [ ]:
X = tf.constant([[1., 2.],
                 [3., 4.]])

W = tf.constant([[ 1., -3.,  5.],
                 [ 2.,  4., -6.]])

Z = tf.matmul(X, W)

print("X shape:", X.shape)
print("W shape:", W.shape)
print("Z = XW:")
print(Z)
print("Z shape:", Z.shape)

## 8. Bias and Broadcasting

A dense layer computes \(Z=XW+b\). A bias vector with one value per neuron is automatically broadcast across all examples.

In [ ]:
b = tf.constant([0.1, -0.2, 0.3])
Z_with_bias = Z + b

print("Z shape:", Z.shape)
print("b shape:", b.shape)
print("Z + b:")
print(Z_with_bias)
print("Result shape:", Z_with_bias.shape)

## 9. Activation Functions

The sigmoid function maps values to `(0,1)`. ReLU is common in modern hidden layers. Softmax converts a vector of scores into class probabilities that sum to 1.

In [ ]:
z = tf.constant([[-2.0, 0.0, 2.0]])

sigmoid_values = tf.math.sigmoid(z)
relu_values = tf.nn.relu(z)
softmax_values = tf.nn.softmax(z)

print("z:", z.numpy())
print("Sigmoid:", sigmoid_values.numpy())
print("ReLU:", relu_values.numpy())
print("Softmax:", softmax_values.numpy())
print("Softmax sum:", tf.reduce_sum(softmax_values, axis=1).numpy())

# Keras Dense Layers

## 10. Creating a Dense Layer

`Dense(units=3, activation="sigmoid")` creates a layer with three neurons. If the input has two features, the weight matrix has shape `(2,3)` and the output has three activations per example.

In [ ]:
from tensorflow.keras.layers import Dense

x = tf.constant([[200.0, 17.0]])

layer_1 = Dense(units=3, activation="sigmoid")
a1 = layer_1(x)

print("Input:")
print(x.numpy())
print("Input shape:", x.shape)
print("\nActivation vector:")
print(a1.numpy())
print("Activation shape:", a1.shape)

The numerical activations above depend on the layer's automatically initialized weights. They are not predetermined values such as `[0.2, 0.7, 0.3]` unless the weights happen to produce those values.

## 11. Inspecting Weights and Biases

After the layer has been called, Keras has created its trainable parameters.

In [ ]:
W1, b1 = layer_1.get_weights()

print("Weight matrix W:")
print(W1)
print("W shape:", W1.shape)

print("\nBias vector b:")
print(b1)
print("b shape:", b1.shape)

print("\nWeights entering neuron 1: W[:, 0]")
print(W1[:, 0])
print("Shape:", W1[:, 0].shape)

print("\nTotal trainable parameters:", layer_1.count_params())

For two input features and three neurons, there are $2\times3=6$ weights and 3 biases, for 9 trainable parameters.

## 12. Reproduce `Dense` Manually

A Keras Dense layer performs the same calculation studied in NumPy:

$
A = g(XW+b).
$

In [ ]:
W1_tf = tf.constant(W1)
b1_tf = tf.constant(b1)

z_manual = tf.matmul(x, W1_tf) + b1_tf
a_manual = tf.math.sigmoid(z_manual)
a_keras = layer_1(x)

print("Manual calculation:")
print(a_manual.numpy())

print("\nKeras Dense output:")
print(a_keras.numpy())

print("\nSame result:", np.allclose(a_manual.numpy(), a_keras.numpy()))

# Building a Neural Network

## 13. `Sequential`

`Sequential` connects layers in order. Here we build a `2 → 3 → 1` binary-classification architecture.

In [ ]:
from tensorflow.keras import Sequential

model = Sequential([
    tf.keras.Input(shape=(2,)),
    Dense(units=3, activation="sigmoid"),
    Dense(units=1, activation="sigmoid")
])

model.summary()

## 14. Forward Propagation Through the Network

For a batch of four examples:

- input: `(4,2)`
- hidden activation: `(4,3)`
- output activation: `(4,1)`

In [ ]:
X_batch = tf.constant([
    [200.0, 17.0],
    [120.0,  5.0],
    [425.0, 20.0],
    [212.0, 18.0]
])

hidden_layer = model.layers[0]
output_layer = model.layers[1]

A1 = hidden_layer(X_batch)
A2 = output_layer(A1)
model_output = model(X_batch)

print("X shape:", X_batch.shape)
print("A1 shape:", A1.shape)
print("A2 shape:", A2.shape)
print("\nA2:")
print(A2.numpy())
print("\nMatches complete model:", np.allclose(A2.numpy(), model_output.numpy()))

Mathematically, this is

$
A^{[1]}=\sigma(XW^{[1]}+b^{[1]}),
\qquad
A^{[2]}=\sigma(A^{[1]}W^{[2]}+b^{[2]}).
$

# Training and Prediction

## 15. `compile()`

`compile()` configures how the model will learn. For binary classification, a common configuration uses Adam and binary cross-entropy.

In [ ]:
train_model = Sequential([
    tf.keras.Input(shape=(2,)),
    Dense(3, activation="sigmoid"),
    Dense(1, activation="sigmoid")
])

train_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully.")

## 16. `fit()` — Training

`fit()` changes the model parameters. During training, TensorFlow performs forward propagation, calculates the loss, uses automatic differentiation/backpropagation to obtain gradients, and updates the weights with the optimizer.

The small dataset below is only for demonstrating the API.

In [ ]:
X_train = np.array([
    [-1.0, -1.0],
    [-1.0,  1.0],
    [ 1.0, -1.0],
    [ 1.0,  1.0],
    [ 0.8,  0.9],
    [-0.8, -0.9]
], dtype=np.float32)

y_train = np.array([[0], [0], [0], [1], [1], [0]], dtype=np.float32)

history = train_model.fit(
    X_train,
    y_train,
    epochs=100,
    verbose=0
)

print("Training complete.")
print("Final loss:", history.history["loss"][-1])
print("Final accuracy:", history.history["accuracy"][-1])

## 17. Plot the Training Loss

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"])
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy loss")
plt.title("Training Loss")
plt.show()

## 18. `predict()` — Inference

`predict()` uses the current learned weights but does not update them.

In [ ]:
probabilities = train_model.predict(X_train, verbose=0)
predictions = (probabilities >= 0.5).astype(int)

print("Probabilities:")
print(probabilities)

print("\nPredicted classes:")
print(predictions)

print("\nTrue labels:")
print(y_train.astype(int))

## 19. `model(x)` vs. `model.predict(x)`

Both perform inference. Directly calling the model returns a TensorFlow tensor; `predict()` is a higher-level inference interface and commonly returns NumPy arrays.

In [ ]:
direct_output = train_model(X_train)
predict_output = train_model.predict(X_train, verbose=0)

print("model(x) type:", type(direct_output))
print("model.predict(x) type:", type(predict_output))
print("Same numerical result:",
      np.allclose(direct_output.numpy(), predict_output))

# Binary vs. Multiclass Classification

## 20. Binary Output

One sigmoid neuron produces one probability per example.

In [ ]:
binary_layer = Dense(1, activation="sigmoid")
example_features = tf.random.normal((5, 15))
binary_probabilities = binary_layer(example_features)

print("Binary output shape:", binary_probabilities.shape)
print(binary_probabilities.numpy())

## 21. Multiclass Output

For general digit recognition with classes 0–9, use 10 output neurons. Softmax makes each row a probability distribution across the 10 classes.

In [ ]:
digit_model = Sequential([
    tf.keras.Input(shape=(784,)),
    Dense(25, activation="sigmoid"),
    Dense(15, activation="sigmoid"),
    Dense(10, activation="softmax")
])

dummy_images = tf.random.uniform((5, 784))
digit_probabilities = digit_model(dummy_images)
predicted_digits = tf.argmax(digit_probabilities, axis=1)

print("Input shape:", dummy_images.shape)
print("Output shape:", digit_probabilities.shape)
print("Probability sums:", tf.reduce_sum(digit_probabilities, axis=1).numpy())
print("Predicted class indices:", predicted_digits.numpy())

# TensorFlow and Vectorization

## 22. Vectorized Dense-Layer Computation

TensorFlow does not need a Python loop over every example and neuron. Matrix multiplication processes the entire batch simultaneously.

In [ ]:
X = tf.constant([
    [ 0.5, -0.2],
    [-0.3,  0.8],
    [ 0.9,  0.4],
    [-0.6, -0.1]
], dtype=tf.float32)

W = tf.constant([
    [ 1.0, -0.8,  0.6],
    [ 0.5,  1.2, -1.0]
], dtype=tf.float32)

b = tf.constant([[0.1, -0.2, 0.3]], dtype=tf.float32)

Z = tf.matmul(X, W) + b
A = tf.math.sigmoid(Z)

print("X:", X.shape)
print("W:", W.shape)
print("b:", b.shape)
print("Z:", Z.shape)
print("A:", A.shape)
print("\nActivations:")
print(A.numpy())

The dimensions show the vectorization directly:

$
(4,2)(2,3)\rightarrow(4,3).
$

All **four examples** and all **three neurons** are processed in the same matrix operation.

## 23. NumPy vs. TensorFlow

The forward mathematics is the same in both libraries.

In [ ]:
X_np = X.numpy()
W_np = W.numpy()
b_np = b.numpy()

A_numpy = 1 / (1 + np.exp(-(X_np @ W_np + b_np)))
A_tensorflow = tf.math.sigmoid(tf.matmul(X, W) + b).numpy()

print("NumPy output:")
print(A_numpy)

print("\nTensorFlow output:")
print(A_tensorflow)

print("\nEquivalent:", np.allclose(A_numpy, A_tensorflow))
print("Maximum absolute difference:",
      np.max(np.abs(A_numpy - A_tensorflow)))

TensorFlow becomes especially useful beyond this forward calculation because it provides automatic differentiation, optimizers, GPU/accelerator support, neural-network layers, training loops, model saving, and deployment tools.

# 24. Complete Example

This final example shows the workflow students will encounter in the Week 7 practical:

$
\text{data}\rightarrow\text{model}\rightarrow\text{compile}\rightarrow
\text{fit}\rightarrow\text{predict}.
$

In [ ]:
# 1. Data: 8 examples, 2 features
X = np.array([
    [-1.0, -1.0],
    [-1.0,  0.5],
    [-0.5, -0.8],
    [ 0.2,  0.1],
    [ 0.5,  0.7],
    [ 0.8,  0.9],
    [ 1.0, -0.2],
    [ 1.2,  1.0]
], dtype=np.float32)

y = np.array([[0], [0], [0], [0], [1], [1], [0], [1]],
             dtype=np.float32)

# 2. Architecture
model = Sequential([
    tf.keras.Input(shape=(2,)),
    Dense(3, activation="sigmoid"),
    Dense(1, activation="sigmoid")
])

# 3. Configure training
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# 4. Train
history = model.fit(X, y, epochs=100, verbose=0)

# 5. Inference
probabilities = model.predict(X, verbose=0)
predictions = (probabilities >= 0.5).astype(int)

print("X shape:", X.shape)
print("Probability shape:", probabilities.shape)
print("\nProbabilities:")
print(np.round(probabilities, 3))
print("\nPredictions:")
print(predictions.T)

# 6. Architecture and parameter counts
model.summary()

# Summary

The most important TensorFlow relationships for Week 7 are:

$
X:(m,n)
$

where $m$ is the number of examples and $n$ is the number of input features.

For a dense layer with $u$ neurons:

$
W:(n,u), \qquad Z=XW+b, \qquad A=g(Z):(m,u).
$

In Keras:

```python
layer = Dense(units=u, activation="sigmoid")
A = layer(X)
```

implements this vectorized calculation.

The main Keras workflow is:

```text
Create data
    ↓
Build model
    ↓
Compile model
    ↓
model.fit(...)       Training: parameters change
    ↓
model.predict(...)   Inference: parameters do not change
```

After working through this notebook, students should be ready to implement the TensorFlow sections of the Week 7 neural-networks practical.